In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
import hyperopt
import time
from hyperopt import hp, fmin, tpe, Trials, partial, STATUS_OK
from hyperopt.early_stop import no_progress_loss
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv('/kaggle/input/london-house-price-prediction-advanced-techniques/train.csv')
test = pd.read_csv('/kaggle/input/london-house-price-prediction-advanced-techniques/test.csv')

In [3]:
train.head()

,ID,fullAddress,postcode,country,outcode,latitude,longitude,bathrooms,bedrooms,floorAreaSqM,livingRooms,tenure,propertyType,currentEnergyRating,sale_month,sale_year,price
0,0,"38 Adelina Grove, London, E1 3AD",E1 3AD,England,E1,51.519406,-0.053261,NaN,3.0,80.0,1.0,Freehold,Semi-Detached House,C,1,1995,77000
1,1,"6 Cleveland Grove, London, E1 4XL",E1 4XL,England,E1,51.521261,-0.053384,2.0,4.0,110.0,1.0,Leasehold,Terrace Property,D,1,1995,89995
2,2,"65 Sanderstead Road, London, E10 7PW",E10 7PW,England,E10,51.569054,-0.034892,1.0,3.0,84.0,1.0,Freehold,Terrace Property,D,1,1995,59000
3,3,"5 Queenswood Gardens, London, E11 3SE",E11 3SE,England,E11,51.564212,0.026292,NaN,2.0,72.0,1.0,Leasehold,Purpose Built Flat,NaN,1,1995,51500
4,4,"12 Woodlands Road, London, E11 4RW",E11 4RW,England,E11,51.563430,0.006260,1.0,3.0,104.0,1.0,Freehold,Mid Terrace House,D,1,1995,63500


In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266325 entries, 0 to 266324
Data columns (total 17 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   ID                   266325 non-null  int64  
 1   fullAddress          266325 non-null  object 
 2   postcode             266325 non-null  object 
 3   country              266325 non-null  object 
 4   outcode              266325 non-null  object 
 5   latitude             266325 non-null  float64
 6   longitude            266325 non-null  float64
 7   bathrooms            217846 non-null  float64
 8   bedrooms             241482 non-null  float64
 9   floorAreaSqM         252519 non-null  float64
 10  livingRooms          229285 non-null  float64
 11  tenure               260604 non-null  object 
 12  propertyType         265817 non-null  object 
 13  currentEnergyRating  209511 non-null  object 
 14  sale_month           266325 non-null  int64  
 15  sale_year        

Instead of using the Dtype to determine whether it is category or numeric , we check them one by one

In [5]:
for i in train.columns:
    print(train[i].value_counts())

ID
266324    1
0         1
1         1
2         1
3         1
         ..
24        1
25        1
26        1
27        1
28        1
Name: count, Length: 266325, dtype: int64
fullAddress
74 Western Beach Apartments, 36 Hanover Avenue, London, E16 1DZ    112
Clarendon Court, 2 Brackley Street, London, EC1Y 0AB                84
Oak Apple Court, Gables Close, London, SE12 0UB                     75
Horley Court, 46 Inverness Terrace, London, W2 3JA                  68
6, 14–16 Church Hill, London, E17 3AG                               59
                                                                  ... 
10 Kent Road, London, W4 5EZ                                         1
9 The Avenue, London, W4 1HA                                         1
81 Edmunds House, Colonial Drive, London, W4 5HA                     1
52 Swyncombe Avenue, London, W5 4DS                                  1
Flat 23, Hazel Court, 1 Hamilton Road, Ealing, W5 2EE                1
Name: count, Length: 118345, d

In [6]:
for i in test.columns:
    print(test[i].value_counts())

ID
282871    1
266325    1
266326    1
282832    1
282833    1
         ..
266331    1
266332    1
266333    1
266334    1
266335    1
Name: count, Length: 16547, dtype: int64
fullAddress
35 Crescent Road, London, E10 5JJ                                                      3
Flat 1, 18 Mosslea Road, London, SE20 7BW                                              3
17 Keel Close, London, SE16 6BX                                                        3
84 Willow Vale, London, W12 0PB                                                        3
First Floor and Second Floor Flat, 134 Blackheath Hill, Greenwich, London, SE10 8AY    3
                                                                                      ..
20 Adley Street, London, E5 0DY                                                        1
50 Queens Grove Road, London, E4 7BT                                                   1
48 Furrow House, 12 Jubilee Avenue, London, E4 9JD                                     1
5 Riverleig


Drop the following fields: ID, fullAddress, postcode, country (only one category), outcode. Keep latitude (numeric), longitude (numeric), bathrooms (category), bedrooms (category), floorAreaSqM (numeric), livingRooms (category), tenure (category), propertyType (category), currentEnergyRating (category), sale_month (numeric), sale_year (numeric), and price (numeric, target).

In [7]:
train.isnull().sum()

ID                         0
fullAddress                0
postcode                   0
country                    0
outcode                    0
latitude                   0
longitude                  0
bathrooms              48479
bedrooms               24843
floorAreaSqM           13806
livingRooms            37040
tenure                  5721
propertyType             508
currentEnergyRating    56814
sale_month                 0
sale_year                  0
price                      0
dtype: int64

For the features 'bathrooms', 'bedrooms', and 'livingRooms', there is no category '0'. Therefore, I decided to fill in missing values with '0'.

In [8]:
train['bathrooms'] = train['bathrooms'].fillna(0)

In [9]:
train['bedrooms'] = train['bedrooms'].fillna(0)

In [10]:
train['livingRooms'] = train['livingRooms'].fillna(0)

 For the features 'tenure', 'propertyType', and 'currentEnergyRating', I will create a new category called 'Unknown' to represent the missing values. Alternatively, filling these missing values with the mode is also acceptable.

In [11]:
train['tenure'] = train['tenure'].fillna('Unknown')

In [12]:
train['propertyType'] = train['propertyType'].fillna('Unknown')

In [13]:
train['currentEnergyRating'] = train['currentEnergyRating'].fillna('Unknown')

For the feature 'floorAreaSqM', I will replace the missing values with the mean.

In [14]:
train['floorAreaSqM'] = train['floorAreaSqM'].fillna(train['floorAreaSqM'].mean())

In [15]:
del train['fullAddress']
del train['postcode']
del train['country']
del train['outcode']
del test['fullAddress']
del test['postcode']
del test['country']
del test['outcode']

Check the training data again to confirm that there are no missing values remaining.

In [16]:
train.isnull().sum()

ID                     0
latitude               0
longitude              0
bathrooms              0
bedrooms               0
floorAreaSqM           0
livingRooms            0
tenure                 0
propertyType           0
currentEnergyRating    0
sale_month             0
sale_year              0
price                  0
dtype: int64

For the testing data, I will apply the same procedures as for the training data. I will ensure that the missing features in the testing data match those in the training data, so no extra work is needed. Fortunately, the missing features in the testing data are the same as those in the training data for this competition.

In [17]:
test.isnull().sum()

ID                        0
latitude                  0
longitude                 0
bathrooms              2624
bedrooms               1375
floorAreaSqM           2006
livingRooms            2095
tenure                  590
propertyType            167
currentEnergyRating    1497
sale_month                0
sale_year                 0
dtype: int64

In [18]:
test['bathrooms'] = test['bathrooms'].fillna(0)
test['bedrooms'] = test['bedrooms'].fillna(0)
test['livingRooms'] = test['livingRooms'].fillna(0)
test['tenure'] = test['tenure'].fillna('Unknown')
test['propertyType'] = test['propertyType'].fillna('Unknown')
test['currentEnergyRating'] = test['currentEnergyRating'].fillna('Unknown')
test['floorAreaSqM'] = test['floorAreaSqM'].fillna(train['floorAreaSqM'].mean())

Check the testing data again to confirm that there are no missing values remaining.

In [19]:
test.isnull().sum()

ID                     0
latitude               0
longitude              0
bathrooms              0
bedrooms               0
floorAreaSqM           0
livingRooms            0
tenure                 0
propertyType           0
currentEnergyRating    0
sale_month             0
sale_year              0
dtype: int64

In [20]:
train['sale_year']

0         1995
1         1995
2         1995
3         1995
4         1995
          ... 
266320    2023
266321    2023
266322    2023
266323    2023
266324    2023
Name: sale_year, Length: 266325, dtype: int64

In [21]:
train['sale_year'].value_counts()

sale_year
2023    33409
2022    26332
2021    19766
2020     9625
2019     8702
2006     8641
2007     8632
2018     8577
2002     8565
1999     8470
2014     8280
2015     8270
2017     8108
2016     8003
2001     7836
2004     7695
2000     7476
2013     7319
1997     7233
1998     7083
2003     7078
2005     6782
1996     5856
2012     5547
2011     5259
2010     5205
1995     4578
2008     4123
2009     3875
Name: count, dtype: int64

In [22]:
train['sale_year'] = train['sale_year']-1995

In [23]:
train['sale_year'].value_counts()

sale_year
28    33409
27    26332
26    19766
25     9625
24     8702
11     8641
12     8632
23     8577
7      8565
4      8470
19     8280
20     8270
22     8108
21     8003
6      7836
9      7695
5      7476
18     7319
2      7233
3      7083
8      7078
10     6782
1      5856
17     5547
16     5259
15     5205
0      4578
13     4123
14     3875
Name: count, dtype: int64

In [24]:
test['sale_year'] = test['sale_year']-1995

In [25]:
def find_season(month):
        season_month_north = {
            12:'Winter', 1:'Winter', 2:'Winter',
            3:'Spring', 4:'Spring', 5:'Spring',
            6:'Summer', 7:'Summer', 8:'Summer',
            9:'Autumn', 10:'Autumn', 11:'Autumn'}
        return season_month_north.get(month)

In [26]:
season_list = []
for month in train['sale_month']:
    season = find_season(month)
    season_list.append(season)
    
train['Season'] = season_list

In [27]:
train['Season'].value_counts()

Season
Summer    74508
Autumn    71182
Spring    61792
Winter    58843
Name: count, dtype: int64

In [28]:
season_list = []
for month in test['sale_month']:
    season = find_season(month)
    season_list.append(season)
    
test['Season'] = season_list

In [29]:
test['Season'].value_counts()

Season
Spring    7082
Winter    5125
Summer    4340
Name: count, dtype: int64

In [30]:
cats = ['bathrooms','bedrooms','livingRooms','currentEnergyRating','tenure','propertyType','Season']
num_cols = ['latitude','longitude','floorAreaSqM','sale_year','sale_month']

In [31]:
train.shape[1]

14

In [32]:
len(cats)+len(num_cols)+2

14

In [33]:
encoder = OrdinalEncoder()
train[cats] = encoder.fit_transform(train[cats])
test[cats] = encoder.transform(test[cats])

In [34]:
train[cats]

,bathrooms,bedrooms,livingRooms,currentEnergyRating,tenure,propertyType,Season
0,0.0,3.0,1.0,2.0,1.0,14.0,3.0
1,2.0,4.0,1.0,3.0,2.0,16.0,3.0
2,1.0,3.0,1.0,3.0,1.0,16.0,3.0
3,0.0,2.0,1.0,7.0,2.0,12.0,3.0
4,1.0,3.0,1.0,3.0,1.0,10.0,3.0
...,...,...,...,...,...,...,...
266320,2.0,2.0,1.0,4.0,2.0,8.0,3.0
266321,1.0,2.0,1.0,2.0,2.0,1.0,3.0
266322,1.0,2.0,1.0,2.0,2.0,12.0,3.0
266323,2.0,2.0,1.0,1.0,2.0,8.0,3.0


In [35]:
scaler = StandardScaler()

In [36]:
train[num_cols] = scaler.fit_transform(train[num_cols].values)

In [37]:
test[num_cols] = scaler.transform(test[num_cols].values)

In [38]:
ID_col = 'ID'
target = 'price'

In [39]:
X_train = train.drop(columns=[ID_col, target]).copy()
X_test = test.drop(columns=[ID_col]).copy()
y_train = train['price'].copy()

DMatrix is an internal data structure that is used by XGBoost, which is optimized for both memory efficiency and training speed.

In [40]:
data_xgb = xgb.DMatrix(X_train,label=y_train,enable_categorical = True)

[Graph]
I wrote some code to generate graphs that will help determine the initial range for 'num_boost_round', 'min_child_weight', and 'lambda'. Since this process takes some time, I will store the code in a markdown cell.

def overfitcheck(result):
    return (result.iloc[-1,2] - result.iloc[-1,0]).min()

train = []
test = []
option = np.arange(10,300,10)
overfit = []
for i in option:
    params = {"max_depth":5,"seed":1412,"eta":0.1, "nthread":16
             }
    result = xgb.cv(params,data_xgb,num_boost_round=i
                ,nfold=5 
                ,seed=1412 
               )
    overfit.append(overfitcheck(result))
    train.append(result.iloc[-1,0])
    test.append(result.iloc[-1,2])
plt.plot(option,test);

plt.plot(option,train);

plt.plot(option,overfit);

plt.plot(option,train, label = "train")
plt.plot(option,test, label = "test")
plt.title("num_boost_round")
plt.legend()
plt.show()

We set the num_boost_round range(50,200,10)

train = []
test = []
option = np.arange(0,7000,70)
overfit = []
for i in option:
    params = {"max_depth":5,"seed":1412,"eta":0.1, "nthread":16
              ,"min_child_weight":i
             }
    result = xgb.cv(params,data_xgb,num_boost_round=50
                ,nfold=5 
                ,seed=1412 
               )
    overfit.append(overfitcheck(result))
    train.append(result.iloc[-1,0])
    test.append(result.iloc[-1,2])
plt.plot(option,test);

plt.plot(option,train);

plt.plot(option,overfit);

plt.plot(option,train, label = "train")
plt.plot(option,test, label = "test")
plt.title("min_child_weight")
plt.legend()
plt.show()

We set the min_child_weight range(50,600,50)

train = []
test = []
lambda_ = np.arange(1,2,0.1)
overfit = []
for i in lambda_:
    params = {"max_depth":5,"seed":1412,"eta":0.1
              ,"lambda":float(i)
             }
    result = xgb.cv(params,data_xgb,num_boost_round=50
                ,nfold=5 
                ,seed=1412 
               )
    overfit.append(overfitcheck(result))
    train.append(result.iloc[-1,0])
    test.append(result.iloc[-1,2])

plt.plot(lambda_,train);

plt.plot(lambda_,test,color="red");

plt.plot(lambda_,overfit,color="orange");

We set the lambda range(0,3,0.2)

I used Hyperopt to search for hyperparameters for the XGBoost algorithm. To manage computational costs, I initially set a smaller num_boost_round and a larger learning rate. After finding a more accurate range for the other hyperparameters, I will gradually increase the num_boost_round and decrease the learning rate.

I will not perform a train-test split because cross-validation will be handled within the hyperparameter searching function.

In [41]:
#First-Round range of hyperparameters

param_grid_simple = {'num_boost_round': hp.quniform("num_boost_round",50,200,10)
                     ,"eta": hp.quniform("eta",0.05,2.05,0.05)
                     ,"colsample_bytree":hp.quniform("colsample_bytree",0.4,1,0.1)
                     ,"lambda":hp.quniform("lambda",0,3,0.2)
                     ,"min_child_weight":hp.quniform("min_child_weight",50,600,50)
                     ,"max_depth":hp.choice("max_depth",range(3,10))
                     ,"subsample":hp.quniform("subsample",0.5,1,0.1)
                     ,"rate_drop":hp.quniform("rate_drop",0.1,1,0.1)
                    }

In [42]:
def hyperopt_objective(params):
    paramsforxgb = {"eta":params["eta"]
                    ,"colsample_bytree":params["colsample_bytree"]
                    ,"lambda":params["lambda"]
                    ,"min_child_weight":params["min_child_weight"]
                    ,"max_depth":int(params["max_depth"])
                    ,"subsample":params["subsample"]
                    ,"rate_drop":params["rate_drop"]
                    ,"nthread":14
                    ,"verbosity":0
                    ,"seed":1412}
    result = xgb.cv(params,data_xgb, seed=1412, metrics=("mae")
                    ,num_boost_round=int(params["num_boost_round"]))
    return result.iloc[-1,2]

In [43]:
def param_hyperopt(max_evals=100):
    
    trials = Trials()
    
    early_stop_fn = no_progress_loss(30)
    
    params_best = fmin(hyperopt_objective
                       , space = param_grid_simple
                       , algo = tpe.suggest
                       , max_evals = max_evals
                       , verbose=True
                       , trials = trials
                       , early_stop_fn = early_stop_fn
                      )
    
    print("\n","\n","best params: ", params_best,
          "\n")
    return params_best, trials

In [44]:
#I used these four lines of code to search for hyperparameters and measure the running time. I ran it five times to obtain five sets of parameters.
#start = time.time()
#params_best, trials = param_hyperopt(100)
#end = time.time()
#print(end - start)

In [45]:
#Here is the output if you run the code,it prompt the best parameters and it takes 406.9175612926483 seconds.

#100%|██████████| 100/100 [06:46<00:00,  4.07s/trial, best loss: 186994.69839028255]

#best params:  {'colsample_bytree': 0.7000000000000001, 'eta': 0.1, 'lambda': 0.4, 'max_depth': 5, 'min_child_weight': 100.0, 'num_boost_round': 180.0, 'rate_drop': 1.0, 'subsample': 0.8} 

#406.9175612926483

# The hp.choice function outputs an index. For example, "max_depth": hp.choice("max_depth", range(3, 10)) means that if 'max_depth' is 5, it actually corresponds to the value 8.


Here is the First round result.

best params:  {'colsample_bytree': 0.7000000000000001, 'eta': 0.1, 'lambda': 0.4, 'max_depth': 5, 'min_child_weight': 100.0, 'num_boost_round': 180.0, 'rate_drop': 1.0, 'subsample': 0.8} 

best params:  {'colsample_bytree': 0.7000000000000001, 'eta': 0.05, 'lambda': 1.2000000000000002, 'max_depth': 4, 'min_child_weight': 200.0, 'num_boost_round': 130.0, 'rate_drop': 0.2, 'subsample': 1.0} 

best params:  {'colsample_bytree': 0.5, 'eta': 0.1, 'lambda': 1.8, 'max_depth': 4, 'min_child_weight': 200.0, 'num_boost_round': 200.0, 'rate_drop': 0.2, 'subsample': 0.8} 

best params: {'colsample_bytree': 0.8, 'eta': 0.1, 'lambda': 0.6000000000000001, 'max_depth': 6, 'min_child_weight': 200.0, 'num_boost_round': 160.0, 'rate_drop': 0.7000000000000001, 'subsample': 1.0} 

best params:  {'colsample_bytree': 0.8, 'eta': 0.1, 'lambda': 1.6, 'max_depth': 5, 'min_child_weight': 100.0, 'num_boost_round': 120.0, 'rate_drop': 0.2, 'subsample': 1.0} 

We adjust the hyperparameter searching range according to the First-round result


In [46]:
#Second-Round range of hyperparameters
param_grid_simple = {'num_boost_round': hp.quniform("num_boost_round",100,500,50)
                     ,"eta": hp.quniform("eta",0.01,0.15,0.02)
                     ,"colsample_bytree":hp.quniform("colsample_bytree",0.6,1,0.1)
                     ,"lambda":hp.quniform("lambda",0,2,0.1)
                     ,"min_child_weight":hp.quniform("min_child_weight",50,250,25)
                     ,"max_depth":hp.choice("max_depth",range(6,10))
                     ,"subsample":hp.quniform("subsample",0.7,1,0.1)
                     ,"rate_drop":hp.quniform("rate_drop",0.1,1,0.1)
                    }

In [47]:
#I used these four lines of code to search for hyperparameters and measure the running time. I ran it five times to obtain five sets of parameters.
#start = time.time()
#params_best, trials = param_hyperopt(100)
#end = time.time()
#print(end - start)

Here is the Second round result.

 best params:  {'colsample_bytree': 0.9, 'eta': 0.04, 'lambda': 1.0, 'max_depth': 3, 'min_child_weight': 50.0, 'num_boost_round': 250.0, 'rate_drop': 0.8, 'subsample': 1.0} 
 
 best params:  {'colsample_bytree': 0.9, 'eta': 0.04, 'lambda': 1.5, 'max_depth': 2, 'min_child_weight': 50.0, 'num_boost_round': 450.0, 'rate_drop': 0.4, 'subsample': 0.9} 
 
 best params:  {'colsample_bytree': 0.9, 'eta': 0.02, 'lambda': 0.2, 'max_depth': 3, 'min_child_weight': 50.0, 'num_boost_round': 500.0, 'rate_drop': 0.8, 'subsample': 1.0}
 
 best params:  {'colsample_bytree': 0.9, 'eta': 0.02, 'lambda': 1.3, 'max_depth': 3, 'min_child_weight': 50.0, 'num_boost_round': 450.0, 'rate_drop': 0.4, 'subsample': 0.9} 
 
 best params:  {'colsample_bytree': 0.8, 'eta': 0.02, 'lambda': 1.7000000000000002, 'max_depth': 3, 'min_child_weight': 50.0, 'num_boost_round': 500.0, 'rate_drop': 0.30000000000000004, 'subsample': 1.0} 

 We adjust the hyperparameter searching range according to the Second-round result

In [48]:
#Third-Round range of hyperparameters
param_grid_simple = {'num_boost_round': hp.quniform("num_boost_round",500,2500,500)
                     ,"eta": hp.quniform("eta",0.01,0.1,0.005)
                     ,"colsample_bytree":hp.quniform("colsample_bytree",0.7,1,0.05)
                     ,"lambda":hp.quniform("lambda",0.8,2,0.1)
                     ,"min_child_weight":hp.quniform("min_child_weight",20,70,5)
                     ,"max_depth":hp.choice("max_depth",range(6,10))
                     ,"subsample":hp.quniform("subsample",0.8,1,0.1)
                     ,"rate_drop":hp.quniform("rate_drop",0.1,1,0.1)
                    }

In [49]:
#I used these four lines of code to search for hyperparameters and measure the running time. I ran it five times to obtain five sets of parameters.
#start = time.time()
#params_best, trials = param_hyperopt(100)
#end = time.time()
#print(end - start)

In [50]:
#Here is the Third round result.
#best params:  {'colsample_bytree': 0.8, 'eta': 0.02, 'lambda': 1.9000000000000001, 'max_depth': 3, 'min_child_weight': 30.0, 'num_boost_round': 500.0, 'rate_drop': 0.4, 'subsample': 0.9} 

#best params:  {'colsample_bytree': 0.9, 'eta': 0.015, 'lambda': 1.8, 'max_depth': 2, 'min_child_weight': 30.0, 'num_boost_round': 1000.0, 'rate_drop': 0.4, 'subsample': 1.0} 

#best params: {'colsample_bytree': 0.8500000000000001, 'eta': 0.01, 'lambda': 1.5, 'max_depth': 3, 'min_child_weight': 20.0, 'num_boost_round': 1000.0, 'rate_drop':0.30000000000000004, 'subsample': 0.8} 

#best params:  {'colsample_bytree': 0.8, 'eta': 0.025, 'lambda': 2.0, 'max_depth': 3, 'min_child_weight': 25.0, 'num_boost_round': 500.0, 'rate_drop': 0.5, 'subsample': 1.0} 

#best params:  {'colsample_bytree': 0.9500000000000001, 'eta': 0.015, 'lambda': 1.4000000000000001, 'max_depth': 3, 'min_child_weight': 20.0, 'num_boost_round': 1500.0, 'rate_drop': 0.1, 'subsample': 0.9} 


In [51]:
# 57%|█████▋    | 57/100 [26:48<20:13, 28.22s/trial, best loss: 170350.25443484596]  
 
#best params: {'colsample_bytree': 0.8, 'eta': 0.025, 'lambda': 2.0, 'max_depth': 3, 'min_child_weight': 25.0, 'num_boost_round': 500.0, 'rate_drop': 0.5, 'subsample': 1.0}

# Leaderboard Score 223218.504742

# We adjust the hyperparameter searching range according to the Third-round 

I noticed that the cross-validation score is not accurate. K-Fold cross-validation is not a suitable choice because the goal is to predict future house prices based on past information. Therefore, we should use a time series split for a more accurate cross-validation score. I rewrote the function for hyperparameter searching, and since the training data is already sorted by sale_year, I don't need to sort the data in the code.

In [52]:
def objective(params):
    model = XGBRegressor(
        n_estimators=int(params['num_boost_round']),
        learning_rate=params['eta'],
        colsample_bytree=params['colsample_bytree'],
        reg_lambda=params['lambda'],
        min_child_weight=params['min_child_weight'],
        max_depth=int(params['max_depth']),
        subsample=params['subsample'],
        rate_drop=params['rate_drop'],
        random_state=1412
    )
    tscv = TimeSeriesSplit(n_splits=5)
    scores = []

    for train_index, test_index in tscv.split(X_train):
        Xtrain, Xtest = X_train.iloc[train_index], X_train.iloc[test_index]
        ytrain, ytest = y_train.iloc[train_index], y_train.iloc[test_index]

        model.fit(Xtrain, ytrain)
        predictions = model.predict(Xtest)
        score = mean_absolute_error(ytest, predictions)
        scores.append(score)

    return {'loss': np.mean(scores), 'status': STATUS_OK}

space = {'num_boost_round': hp.quniform("num_boost_round",500,2000,250)
                     ,"eta": hp.quniform("eta",0.005,0.03,0.005)
                     ,"colsample_bytree":hp.quniform("colsample_bytree",0.75,1,0.05)
                     ,"lambda":hp.quniform("lambda",0.8,2.5,0.2)
                     ,"min_child_weight":hp.quniform("min_child_weight",15,50,2)
                     ,"max_depth":hp.choice("max_depth",range(6,10))
                     ,"subsample":hp.quniform("subsample",0.8,1,0.1)
                     ,"rate_drop":hp.quniform("rate_drop",0.1,0.6,0.05)
                    }



In [53]:
#I used these three lines of code to search for hyperparameters and measure the running time
#trials = Trials()
#best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=50, trials=trials)
#print("Best hyperparameters:", best)

Here is the output from the last round. We found that the cross-validation score has become similar to the leaderboard score!

100%|██████████| 50/50 [11:19<00:00, 13.59s/trial, best loss: 231668.4993295123] 
Best hyperparameters: {'colsample_bytree': 0.75, 'eta': 0.01, 'lambda': 1.4000000000000001, 'max_depth': 3, 'min_child_weight': 46.0, 'num_boost_round': 750.0, 'rate_drop': 0.35000000000000003, 'subsample': 1.0}0

Leaderboard score274412.9659012

Since the leaderboard score is worse than the hyperparameters searched in round three, I decided to use the parameters from round three to train the model and submit the results.}

After we get the hyperparameter, we train the model using the training set. Then, we use the model to predict the price of the testing set.

In [54]:
model_xgb = XGBRegressor(num_boost_round =500,enable_categorical=True,colsample_bytree=0.8,eta=0.025,reg_lambda=2.0,max_depth=9,min_child_weight=25,rate_drop=0.5,subsample=1.0)

In [55]:
model_xgb.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eta=0.025, eval_metric=None,
             feature_types=None, feature_weights=None, gamma=None,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=9, max_leaves=None,
             min_child_weight=25, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None, ...)

In [56]:
#encoder = OrdinalEncoder()
#X_test[cats] = encoder.fit_transform(X_test[cats])

In [57]:
pred_xgb = model_xgb.predict(X_test)

In [58]:
sub = pd.read_csv("/kaggle/input/london-house-price-prediction-advanced-techniques/sample_submission.csv")
sub.price = pred_xgb
sub.to_csv("submission.csv",index=False)
print("Sub shape:",sub.shape)
sub.head()

Sub shape: (16547, 2)


,ID,price
0,266325,453668.0000
1,266326,388921.6875
2,266327,481733.6875
3,266328,482086.6250
4,266329,546307.6875


In [59]:
# Leaderboard Score 223218.504742
